# EMP5027 Lecture 6: Setup (Run This First if You Are Not Using Google Colab)

The three deep learning notebooks (6a EuroSAT, 6b PlantVillage, 6c DeepWeeds) are built to run in **Google Colab**, where TensorFlow is already installed and a free GPU is one click away. If you are using Colab, **you can skip this notebook entirely**.

If you would rather run everything on your own laptop, this notebook gets you ready in four short steps:

1. **Check** your computer and Python version.
2. **Install** TensorFlow and the other packages (using the GPU version only where one can actually work).
3. **Verify** that TensorFlow runs, and see whether it found a GPU.
4. **Pre-download** the three datasets, so you do not have to wait for them in class.

**Do I need a GPU?** No. Everything in these notebooks runs on a normal laptop CPU. A GPU simply makes training faster. Training will be slower on a CPU, but the notebooks use small images, early stopping and capped epochs, so it stays manageable. A "no GPU found" message at the end is not an error.

**Which computers can use a GPU?**

| Your computer | What you get |
|---|---|
| Linux with an NVIDIA GPU | GPU (installed automatically below) |
| Windows with an NVIDIA GPU | CPU by default. TensorFlow stopped supporting native Windows GPUs after version 2.10, so a GPU needs WSL2 (see the troubleshooting section at the end) |
| Mac (Intel or Apple Silicon) | CPU. TensorFlow has no official GPU support on macOS |
| Anything else | CPU |

## Before you start: use a fresh environment (recommended)

Installing TensorFlow into an environment you already use for other work is the most common cause of version conflicts. A clean environment avoids the problem. Run these in an **Anaconda Prompt (Windows)** or **Terminal (Mac/Linux)**, not in this notebook:

```
conda create -n emp5027-dl python=3.11 -y
conda activate emp5027-dl
pip install jupyterlab ipykernel
jupyter lab
```

Then open this notebook from inside that JupyterLab. TensorFlow currently supports Python 3.10 to 3.13, and 3.11 is a safe choice. TensorFlow's own installation guide recommends installing it with `pip`, even inside a conda environment, which is why the install step below uses `pip`.

## Step 1: Check your setup

This cell only looks around and prints what it finds. It does not install or change anything.

In [ ]:
import sys, platform, shutil, importlib.util

# sys.modules only contains "google.colab" inside a Colab runtime, so this is a
# reliable way to tell where we are running without asking you to set a flag yourself
IN_COLAB = "google.colab" in sys.modules
SYSTEM = platform.system()        # "Windows", "Darwin" (this is macOS) or "Linux"
MACHINE = platform.machine()      # for example "x86_64" or "arm64" (Apple Silicon)
PY = sys.version_info

# shutil.which looks for a program on your PATH. nvidia-smi is installed along with
# NVIDIA's GPU drivers, so finding it is a quick, safe hint that an NVIDIA GPU is present
HAS_NVIDIA = shutil.which("nvidia-smi") is not None

print(f"Python:          {PY.major}.{PY.minor}.{PY.micro}")
print(f"Operating system: {SYSTEM} ({MACHINE})")
print(f"Running in Colab: {IN_COLAB}")
print(f"NVIDIA GPU driver found: {HAS_NVIDIA}")
print(f"TensorFlow already installed: {importlib.util.find_spec('tensorflow') is not None}")

# TensorFlow only publishes builds for a range of Python versions, so warn early
if not (3, 10) <= (PY.major, PY.minor) <= (3, 13):
    print("\nWARNING: this Python version is outside the 3.10 to 3.13 range TensorFlow supports.")
    print("Create a fresh environment with Python 3.11 (see the box above) before continuing.")

# Decide which TensorFlow build to install. The CUDA build only makes sense on Linux
# with an NVIDIA GPU. On Windows and macOS we use the plain CPU-capable build.
USE_CUDA = (SYSTEM == "Linux") and HAS_NVIDIA
if IN_COLAB:
    print("\nPlan: Colab already has TensorFlow, so only the extra packages will be installed.")
elif USE_CUDA:
    print("\nPlan: install the NVIDIA GPU build of TensorFlow (tensorflow[and-cuda]).")
else:
    print("\nPlan: install the standard TensorFlow build, which runs on the CPU.")

## Step 2: Install the packages

This can take several minutes, because TensorFlow is a large download. You will see a lot of text scroll past, which is normal. When it finishes, **you must restart the kernel** (the next cell tells you how), otherwise Python may keep using the old state and the install will look as if it failed.

We use `%pip` rather than `!pip` because `%pip` always installs into the same Python environment this notebook is running in. Plain `!pip` can quietly install into a different Python on your machine, which is a classic source of "I installed it but it still says module not found".

In [ ]:
# Packages the three deep learning notebooks need besides TensorFlow itself
EXTRA_PACKAGES = ["tensorflow-datasets", "scikit-learn", "seaborn", "matplotlib"]

if IN_COLAB:
    packages = EXTRA_PACKAGES                      # Colab already ships TensorFlow
elif USE_CUDA:
    packages = ["tensorflow[and-cuda]"] + EXTRA_PACKAGES
else:
    packages = ["tensorflow"] + EXTRA_PACKAGES

print("Installing:", packages)

# Join the package names into one string, each wrapped in quotes so that the square
# brackets in tensorflow[and-cuda] are not misread by the command line
PIP_ARGS = " ".join(f'"{p}"' for p in packages)

# The {PIP_ARGS} part is filled in by Python before pip runs, so the plan decided in
# Step 1 is what actually gets installed
%pip install -q {PIP_ARGS}

### Now restart the kernel

In the menu choose **Kernel > Restart Kernel** (in Colab: **Runtime > Restart session**). Do **not** re-run the install cell. Carry on from Step 3 below.

## Step 3: Verify that TensorFlow works

This imports TensorFlow, lists any GPUs it can see, and times a small matrix multiplication so you can see your machine doing real work.

**Do not worry about start-up messages.** On a computer without an NVIDIA GPU you will probably see lines beginning `I0000` or `E0000` mentioning CUDA, drivers or `cuInit`. They look alarming but are harmless. They simply mean TensorFlow looked for a GPU, did not find one, and is carrying on with the CPU. The lines that matter are the ones this cell prints itself.

In [ ]:
import os, time

# TensorFlow prints a lot of low-level start-up messages by default. Level 2 reduces
# the routine ones but still shows genuine warnings and errors. This must be set BEFORE
# TensorFlow is imported, which is why it comes first
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU found: {gpus[0].name}. Training will use it automatically.")
else:
    print("No GPU found. TensorFlow will use the CPU, which is completely fine for these notebooks.")

# A quick benchmark: multiply two 2000 x 2000 matrices a few times. The first call is
# slow because TensorFlow is warming up, so it is run once before the timed loop
a = tf.random.normal([2000, 2000])
b = tf.random.normal([2000, 2000])
_ = tf.matmul(a, b).numpy()

start = time.time()
for _ in range(5):
    _ = tf.matmul(a, b).numpy()
elapsed = time.time() - start
print(f"5 matrix multiplications took {elapsed:.2f} s on {'GPU' if gpus else 'CPU'}")

## Step 4: Pre-download the datasets (optional, but recommended)

The deep learning notebooks fetch their data with `tfds.load(...)` (TensorFlow Datasets). The first time you call it, the dataset is **downloaded and saved to a folder on your computer**. After that, every later run (including after you close and reopen the notebook, or restart your laptop) reuses the saved copy, so the download only ever happens once. You do not need to do anything with file paths yourself.

By default the saved copy lives in a folder called `tensorflow_datasets` inside your home folder. The cell below prints the exact location on your machine.

Why pre-download here? The datasets are large (EuroSAT is the smallest, while PlantVillage and DeepWeeds are several hundred MB each), and a classroom wifi network is a poor place to fetch them all at once. Running this cell at home means the notebooks in class start straight away. The cell also fetches the pretrained **MobileNetV2** weights used in the transfer learning sections, which would otherwise download in the middle of the practical.

If a download fails, the cell prints a message and moves on to the next one. You can simply run it again later, because anything already downloaded is skipped.

In [ ]:
import shutil
import tensorflow_datasets as tfds

# Set any of these to False to skip a dataset (for example, to save disk space)
DATASETS = {
    "eurosat": True,         # Notebook 6a, satellite land-cover
    "plant_village": True,   # Notebook 6b, plant disease
    "deep_weeds": True,      # Notebook 6c, weed species
}

# Where TensorFlow Datasets keeps its saved copies. The expanduser call turns "~" into
# your actual home folder, so the printed path is the real one on your machine
data_root = os.path.expanduser(os.environ.get("TFDS_DATA_DIR", "~/tensorflow_datasets"))
os.makedirs(data_root, exist_ok=True)
free_gb = shutil.disk_usage(data_root).free / 1e9
print(f"Datasets are stored in: {data_root}")
print(f"Free disk space there:  {free_gb:.1f} GB\n")

for name, wanted in DATASETS.items():
    if not wanted:
        print(f"{name}: skipped")
        continue
    try:
        builder = tfds.builder(name)
        # download_and_prepare is a no-op if the data is already saved, so re-running is safe
        builder.download_and_prepare()
        print(f"{name}: ready ({builder.info.splits['train'].num_examples} training images)\n")
    except Exception as e:
        print(f"{name}: could not download ({type(e).__name__}).")
        print("  Check your internet connection (university or public wifi sometimes blocks")
        print("  these downloads), then run this cell again.\n")

In [ ]:
from tensorflow import keras

# The transfer learning sections of notebooks 6a, 6b and 6c load MobileNetV2 with weights
# pretrained on ImageNet. Keras downloads those weights once and keeps them in
# ~/.keras. The three notebooks use image sizes of 96, 128 and 160 pixels, and Keras
# fetches a slightly different weights file for each size, so we request all three here
for size in (96, 128, 160):
    try:
        keras.applications.MobileNetV2(
            input_shape=(size, size, 3), include_top=False, weights="imagenet"
        )
        print(f"MobileNetV2 weights for {size}x{size} images: ready")
    except Exception as e:
        print(f"MobileNetV2 weights for {size}x{size}: could not download ({type(e).__name__}).")
        print("  Check your internet connection and run this cell again.")

## You are ready

Open **6a (EuroSAT)**, **6b (PlantVillage)** or **6c (DeepWeeds)** and run them from the top. Each notebook will detect that it is not in Colab and carry on locally, and the data you downloaded above will load from disk in seconds.

## Troubleshooting

**"No GPU found."** This is expected on most laptops and is not an error. Everything still runs on the CPU.

**`ModuleNotFoundError: No module named 'tensorflow'` straight after installing.** Restart the kernel (Kernel > Restart Kernel). If it persists, the notebook is running in a different Python environment from the one you installed into. Run `import sys` and then `print(sys.executable)` in a cell to see which Python the notebook is using, and check it points inside your `emp5027-dl` environment.

**Pip reports version conflicts.** This usually means TensorFlow is being installed into an environment that already contains other packages. Create the fresh `emp5027-dl` environment described at the top and start again there. Installing everything into one new environment is far more reliable than trying to repair an old one.

**Training feels slow on a CPU.** That is normal. Try running the notebooks in Google Colab with a free GPU (Runtime > Change runtime type > GPU). If you must stay on your laptop, lower `EPOCHS` in the training cells and close other heavy programs while training runs.

**Downloads fail or stall.** University and public wifi sometimes block the sites these datasets are hosted on. Try a different network (for example, your phone's hotspot), then re-run the download cells. Anything already saved is skipped.

**Running out of disk space.** Set the datasets you do not need to `False` in the download cell, or point TensorFlow Datasets at a drive with more room by setting the `TFDS_DATA_DIR` environment variable before starting Jupyter.

**I have a Windows laptop with an NVIDIA GPU and want to use it.** TensorFlow stopped supporting GPUs on native Windows after version 2.10, so the route is WSL2 (Windows Subsystem for Linux). Install WSL2 with an Ubuntu distribution, create the conda environment inside it, and this notebook will detect Linux plus an NVIDIA driver and install the GPU build automatically. This is optional, and Colab is a much quicker route to a GPU.

**Further reading:** TensorFlow's own installation guide at https://www.tensorflow.org/install/pip covers every platform in detail.